# Stop Your AI Agent from Forgetting User Preferences: Key-Value Memory (Agent State)

A brand-new user arrives. The agent knows nothing about them. They search flights, book one, and ask for a recommendation — and whether the agent can answer *"based on what you know about me"* **after a restart** depends on ONE design decision: **where memory lives**.

One clarification up front, because this trips people up: Strands keeps the full conversation history (`agent.messages`) between calls on the same agent instance, so *within* a session even a memory-less agent "remembers" — the booking is still in the transcript. The real failure is that the transcript is the **only** place the preference exists: nothing structured is learned, and the moment the process restarts (every new request in production), it's all gone.

The simplest agent memory is a **key-value store**: structured facts under named keys, no embeddings, no similarity search. Strands calls it [**agent state**](https://strandsagents.com/docs/user-guide/concepts/agents/state/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) — "key-value storage for stateful information that exists outside of the conversation context". This notebook climbs its durability ladder one rung at a time:

| Test | Memory wiring | Survives |
|------|---------------|----------|
| 1 | none (transcript only) | turns in one process — nothing structured, nothing after a restart |
| 2 | `agent.state` | turns, as a structured profile (within the process) |
| 3 | + `FileSessionManager` | restarts (local disk) |
| 4 | + `S3SessionManager` | restarts in the cloud (Amazon S3 — plain JSON objects, no vectors) |

*(The research literature calls the cross-session loss in Test 1 "memory decay".)*

The tools call **live APIs** — real flight offers from the [Duffel](https://duffel.com) sandbox and real climate data from [Open-Meteo](https://open-meteo.com) — so nothing is hardcoded. But the APIs are scenery: **the experiment is memory**.

Based on:
- [MemoryOS of AI Agent](https://arxiv.org/abs/2506.06326) — Kang et al., 2025
- [Cognitive Memory in Large Language Models](https://arxiv.org/abs/2504.02441) — Shan et al., 2025

This demo uses Strands Agents. The pattern carries over to other agent frameworks.

## Install dependencies

Run once (or from a terminal: `uv venv && uv pip install -r requirements.txt`).

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Configure credentials

- **`OPENAI_API_KEY`** — the model (or switch to Amazon Bedrock in the model cell).
- **`DUFFEL_API_KEY`** — free flight-search sandbox token from [app.duffel.com](https://app.duffel.com) (More → Developers → Access tokens).
- **`SESSIONS_BUCKET`** *(optional, Test 4 only)* — an S3 bucket you own, plus AWS credentials (`aws configure`). Without it, Tests 1-3 still run.

The climate API (Open-Meteo) needs no key.

In [2]:
import os

# python-dotenv loads both keys from a local .env file, so credentials
# never live inside the notebook.
from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env (or switch to Bedrock in the model cell)'
assert os.getenv('DUFFEL_API_KEY'), 'Set DUFFEL_API_KEY in .env — free token at https://app.duffel.com'

## The model

One model object, reused by every agent. The only cell you touch to change providers.

In [3]:
# OTEL_SDK_DISABLED silences OpenTelemetry tracing noise in notebook output.
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id='gpt-4o-mini')  # api_key read from OPENAI_API_KEY

# Amazon Bedrock instead (no OpenAI key; uses your AWS credentials):
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

# Role and approach only — each tool's purpose lives in its own docstring,
# so the prompt never re-describes the tools.
SYSTEM_PROMPT = (
    'You are a flight booking assistant for a returning traveler. '
    'Personalize recommendations using what you know about the user. '
    'Be concise — answer in 2-3 sentences maximum.'
)

# The same conversation drives every test, so the ONLY variable is memory.
TURN_1 = 'Find me flights from JFK to Paris CDG on 2026-09-15, business class.'
TURN_2 = 'Book the cheapest business option.'
TURN_3 = ('Now I need Paris CDG to Tokyo Haneda on 2026-09-22 — '
          'what do you recommend based on what you know about me?')

---
## Test 1 — No memory tools: the transcript is the only memory, and it dies with the process

The stateless tools search and book **real** Duffel offers, but they have no way to write state — plain `@tool` functions with no `ToolContext`.

**Watch two things:**

1. **Turn 3 is personalized anyway.** Strands keeps the whole conversation in `agent.messages` between calls, so "business class" is still in the transcript and the model uses it. Don't let this fool you into thinking the agent *learned* something.
2. **Then the restart.** A brand-new agent instance (what every new process or request is in production) gets the same turn 3 with an empty transcript — and has nothing to base an answer on. No profile was ever written: `user_preferences` stays `None` the whole time.

In [4]:
# Agent is the Strands agent loop: model + tools until the answer is done.
from strands import Agent

# The stateless variants: same Duffel-backed logic, no access to agent.state.
from tools import search_flights_stateless, book_flight_stateless

agent_stateless = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_flights_stateless, book_flight_stateless],
    callback_handler=None,
)

for turn in (TURN_1, TURN_2, TURN_3):
    resp = agent_stateless(turn)
    print(f'User: {turn}\nAgent: {str(resp).strip()[:220]}\n')

print('user_preferences after 3 turns:', agent_stateless.state.get('user_preferences'))
print(f'messages in transcript: {len(agent_stateless.messages)} — the booking lives ONLY here')

# The restart: a NEW agent instance = a new process/request in production.
# No session manager, so the transcript is gone — turn 3 arrives with no context.
agent_restarted = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_flights_stateless, book_flight_stateless],
    callback_handler=None,
)
resp = agent_restarted(TURN_3)
print(f'\n[after restart] User: {TURN_3}\n[after restart] Agent: {str(resp).strip()[:220]}')

User: Find me flights from JFK to Paris CDG on 2026-09-15, business class.
Agent: Here are some business class flight options from JFK to Paris CDG on September 15, 2026:

1. **Duffel Airways**: $1386.52, non-stop, departing at 10:50 AM, arriving at 01:11 AM (next day).
2. **Icelandair**: $1392.10, 1 



User: Book the cheapest business option.
Agent: Your flight from JFK to Paris CDG has been successfully booked with Duffel Airways for $1386.52. Enjoy your trip!



User: Now I need Paris CDG to Tokyo Haneda on 2026-09-22 — what do you recommend based on what you know about me?
Agent: For your flight from Paris CDG to Tokyo Haneda on September 22, 2026, here are some business class options:

1. **British Airways**: $2294.11, non-stop, departing at 4:15 PM, arriving at 12:46 PM (next day).
2. **America

user_preferences after 3 turns: None
messages in transcript: 12 — the booking lives ONLY here



[after restart] User: Now I need Paris CDG to Tokyo Haneda on 2026-09-22 — what do you recommend based on what you know about me?
[after restart] Agent: For your flight from Paris CDG to Tokyo Haneda on September 22, 2026, I recommend the **cheapest option** with American Airlines priced at **$358.71**. It departs at 16:15 and arrives the next day at 12:46, with a direct


The booking happened, the money was "spent" — and everything the agent "knew" was an unstructured transcript. Within the session it personalized (the transcript carried it); after the restart the same question got a generic answer, because `user_preferences` was `None` all along and nothing else survived. The transcript is also fragile *within* long sessions: a sliding window or summarization trims old messages, and the booking scrolls out with them.

---
## Test 2 — `agent.state`: the booking action teaches the agent

Same conversation, but now the tools carry `@tool(context=True)`: Strands injects a `ToolContext`, and `tool_context.agent.state` is a key-value store that lives **outside the conversation context**. `book_flight` writes what the choice reveals (cabin, stops, price band, carrier); `search_flights` reads it back and **ranks real offers by the learned profile** — deterministic code, not the model hopefully re-reading the transcript.

In [5]:
# json pretty-prints what the agent learned.
import json

# SlidingWindowConversationManager caps message history — the profile belongs
# in agent.state, not in an ever-growing transcript.
from strands.agent.conversation_manager import SlidingWindowConversationManager

# The stateful variants + profile introspection + a real climate tool.
from tools import search_flights, book_flight, get_user_profile, best_time_to_visit

agent_stateful = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[search_flights, book_flight, get_user_profile, best_time_to_visit],
    callback_handler=None,
)

for turn in (TURN_1, TURN_2, TURN_3):
    resp = agent_stateful(turn)
    print(f'User: {turn}\nAgent: {str(resp).strip()[:220]}\n')

print('user_preferences:', json.dumps(agent_stateful.state.get('user_preferences'), indent=1))

User: Find me flights from JFK to Paris CDG on 2026-09-15, business class.
Agent: Here are some business class flight options from JFK to Paris CDG on September 15, 2026:

1. **Icelandair**: Depart 8:30 PM, arrive next day 6:20 AM with one stop. Price: $1,392.10
2. **Icelandair**: Depart 11:10 PM, arr



User: Book the cheapest business option.
Agent: Your flight from JFK to Paris CDG on September 15, 2026, with Icelandair in business class has been confirmed at a price of $1,392.10. Your preferences have been updated to reflect your choice. Safe travels!



User: Now I need Paris CDG to Tokyo Haneda on 2026-09-22 — what do you recommend based on what you know about me?
Agent: Here are your recommended business class flight options from Paris CDG to Tokyo Haneda on September 22, 2026:

1. **Iberia**: Non-stop, depart 4:15 PM, arrive next day 12:46 PM. Price: $2,257.90
2. **British Airways**: N

user_preferences: {
 "preferred_cabin": "business",
 "prefers_nonstop": false,
 "carriers_flown": [
  "Icelandair"
 ],
 "typical_price": {
  "min": 1392.1,
  "max": 1392.1
 }
}


The difference from Test 1 is not the wording of turn 3 — it's **what exists afterwards**: a structured, inspectable profile in `agent.state`, built from the booking action instead of a form. The search tool ranks offers with it in code, `get_user_profile` can show it, and (next test) a session manager can persist it. In Test 1 the same knowledge existed only as prose inside the transcript.

The climate tool is available too — try `agent_stateful("When is the best time of year to visit Tokyo?")` — the agent answers with real historical monthly averages.

---
## Test 3 — `FileSessionManager`: the profile survives a restart

`agent.state` lives in the Python process — restart the app and it's gone, exactly like the transcript in Test 1. `FileSessionManager` persists state to disk keyed by `session_id` — a **brand-new agent instance** with the same id starts already knowing the user. (In production you'd use `S3SessionManager` — same interface, S3 storage.)

In [6]:
# FileSessionManager persists agent.state to disk, keyed by session_id.
from strands.session import FileSessionManager

# shutil only cleans up the session files at the end of the demo.
import shutil

session_id = 'traveler-demo'
storage_dir = os.path.join(os.path.abspath('.'), 'sessions')

def make_agent():
    return Agent(
        model=MODEL,
        system_prompt=SYSTEM_PROMPT,
        conversation_manager=SlidingWindowConversationManager(window_size=40),
        tools=[search_flights, book_flight, get_user_profile, best_time_to_visit],
        session_manager=FileSessionManager(session_id=session_id, storage_dir=storage_dir),
        callback_handler=None,
    )

# Session A: the user books, the profile is built.
agent_a = make_agent()
agent_a(TURN_1)
agent_a(TURN_2)
prefs_a = agent_a.state.get('user_preferences')
print('Session A learned:', json.dumps(prefs_a))

# Session B: NEW agent instance, same session_id — this is the restart.
agent_b = make_agent()
prefs_b = agent_b.state.get('user_preferences')
print('Session B restored:', json.dumps(prefs_b))
print('State survived restart:', prefs_a == prefs_b and prefs_b is not None)

resp = agent_b(TURN_3)
print(f'\nUser: {TURN_3}\nAgent: {str(resp).strip()[:250]}')

shutil.rmtree(storage_dir, ignore_errors=True)

Session A learned: {"preferred_cabin": "business", "prefers_nonstop": true, "carriers_flown": ["Iberia"], "typical_price": {"min": 1360.92, "max": 1360.92}}
Session B restored: {"preferred_cabin": "business", "prefers_nonstop": true, "carriers_flown": ["Iberia"], "typical_price": {"min": 1360.92, "max": 1360.92}}
State survived restart: True



User: Now I need Paris CDG to Tokyo Haneda on 2026-09-22 — what do you recommend based on what you know about me?
Agent: Here are some business class options from Paris CDG to Tokyo Haneda on September 22, 2026, based on your preferences:

1. **Iberia**: $2,279.47, non-stop, departs at 4:15 PM, arrives the next day at 12:46 PM. [Book this flight](#).
2. **American Airl


---
## Test 4 — `S3SessionManager`: the same ladder, one rung up, in the cloud

`S3SessionManager` is the same interface as `FileSessionManager`, but the session persists as **plain JSON objects in Amazon S3** — regular S3, no vectors, no embeddings. **When to choose it over local disk:** nothing to provision or mount (a durable filesystem on Lambda/Fargate means wiring up EFS), and any compute instance can restore the session — the state stops being tied to one machine. Same test: a brand-new agent instance with the same `session_id` restores the profile, this time from a bucket.

In [7]:
# S3SessionManager: identical interface, S3 storage — the production rung.
from strands.session import S3SessionManager

# boto3 is only used to clean up the demo's session objects at the end.
import boto3

bucket = os.getenv('SESSIONS_BUCKET')
assert bucket, 'Set SESSIONS_BUCKET in .env to run this test (created if it does not exist)'

# Self-provisioning: if the bucket doesn't exist, create it (private, public access
# blocked). us-east-1 must NOT send a LocationConstraint; other regions require it.
s3 = boto3.client('s3')
try:
    s3.head_bucket(Bucket=bucket)
except s3.exceptions.ClientError:
    region = s3.meta.region_name or 'us-east-1'
    params = {'Bucket': bucket}
    if region != 'us-east-1':
        params['CreateBucketConfiguration'] = {'LocationConstraint': region}
    s3.create_bucket(**params)
    s3.put_public_access_block(Bucket=bucket, PublicAccessBlockConfiguration={
        'BlockPublicAcls': True, 'IgnorePublicAcls': True,
        'BlockPublicPolicy': True, 'RestrictPublicBuckets': True})
    print(f'created private bucket s3://{bucket} in {region}')

s3_session_id = 'traveler-demo-s3'
prefix = 'kv-memory-demo'

def make_s3_agent():
    return Agent(
        model=MODEL,
        system_prompt=SYSTEM_PROMPT,
        conversation_manager=SlidingWindowConversationManager(window_size=40),
        tools=[search_flights, book_flight, get_user_profile, best_time_to_visit],
        session_manager=S3SessionManager(session_id=s3_session_id, bucket=bucket, prefix=prefix),
        callback_handler=None,
    )

agent_a = make_s3_agent()
agent_a(TURN_1)
agent_a(TURN_2)
prefs_a = agent_a.state.get('user_preferences')
print('Session A learned:', json.dumps(prefs_a))

agent_b = make_s3_agent()  # new instance, same session_id — restored FROM S3
prefs_b = agent_b.state.get('user_preferences')
print(f'Session B restored from s3://{bucket}/{prefix}:', json.dumps(prefs_b))
print('State survived restart:', prefs_a == prefs_b and prefs_b is not None)

# Clean up the demo's session objects so reruns start empty.
listed = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
keys = [{'Key': o['Key']} for o in listed.get('Contents', [])]
if keys:
    s3.delete_objects(Bucket=bucket, Delete={'Objects': keys})

Session A learned: {"preferred_cabin": "business", "prefers_nonstop": true, "carriers_flown": ["British Airways"], "typical_price": {"min": 1374.36, "max": 1374.36}}


Session B restored from s3://agent-memory-demo-sessions-<account-id>/kv-memory-demo: {"preferred_cabin": "business", "prefers_nonstop": true, "carriers_flown": ["British Airways"], "typical_price": {"min": 1374.36, "max": 1374.36}}
State survived restart: True


---
## Summary

| Test | Memory wiring | Structured profile | Survives restart |
|------|---------------|--------------------|------------------|
| 1 | none (transcript only) | No | No |
| 2 | `agent.state` | Yes | No |
| 3 | + `FileSessionManager` | Yes | Yes (local disk) |
| 4 | + `S3SessionManager` | Yes | Yes (Amazon S3) |

**Key insight:** memory is wiring, not model. Test 1 looked fine within the session — the transcript carried the preference — but nothing structured was ever learned, and one restart erased everything. Tests 2-4 made the same knowledge explicit (`agent.state`) and then durable, one rung at a time: process → disk → S3.

**Next:** [Demo 02 — Vector Memory](../02-vector-memory-demo/) puts the JSON into a vector store and retrieves by meaning.